# Model preprocessing — length of stay

This notebook creates a leakage-conscious dataset for predicting total hospital length of stay (LOS). It assumes prediction occurs before the intervention, using only information expected to be available for planning. It does not encode categorical variables; encoding belongs inside the model pipeline and must be fitted on the training split only.

Output: `resources/model_los_dataset.parquet`

> Privacy: row-level identifiers are retained only as `split_patient_id` for split auditing in the ignored `resources/` directory. They must never be used as model features or displayed.

## Prediction-time assumptions and leakage controls

The target is the inclusive calendar-day stay `duree_sejour_corrigee`. The hospital should confirm that this is the intended operational definition: it differs from the recorded stay field in a substantial number of cases.

The feature set deliberately excludes discharge date, every stay-duration field, the stay inconsistency flag, operating-room timestamps and durations, GHM/GHS groupings, case ID, and patient ID. Principal CIM and CCAM codes are included under the assumption that planned diagnosis and procedure codes are available before surgery. Set `INCLUDE_PLANNED_CLINICAL_CODES = False` if that is not true locally.

In [ ]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

DATA_FILE = Path("resources/donnees_bloc_pretraitees.parquet")
OUTPUT_FILE = Path("resources/model_los_dataset.parquet")
INCLUDE_PLANNED_CLINICAL_CODES = True
MIN_CATEGORY_COUNT = 10
VALIDATION_START = pd.Timestamp("2021-01-01")
TEST_START = pd.Timestamp("2022-01-01")
MISSING_LABEL = "__MISSING__"
RARE_LABEL = "__RARE__"

assert DATA_FILE.exists(), (
    f"Missing {DATA_FILE}. Run EDA_donees_bloc.ipynb from top to bottom first."
)

In [ ]:
df = pd.read_parquet(DATA_FILE, engine="pyarrow")
required_columns = {
    "id_patient", "date_inter", "age_years", "sexe", "interv_type",
    "anesth_type", "anesth_loco_reg", "praticien", "nom_chir",
    "cim_diag_pr", "ccam_1", "duree_sejour_corrigee",
}
missing_columns = required_columns.difference(df.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"

print(f"Loaded {len(df):,} cases and {df.shape[1]} columns.")
print(
    "Recorded versus corrected LOS disagreements: "
    f"{df['duree_sejour_incoherente'].sum():,}"
)

## 1. Define and validate the target

No target values are winsorized or silently removed as outliers. Long stays can be clinically valid and should be reviewed with domain experts. The notebook provides the raw positive-day target, a `log1p` variant, and a one-day versus multi-day target for a possible two-stage model.

In [ ]:
target = pd.to_numeric(df["duree_sejour_corrigee"], errors="coerce")
valid_target_mask = target.notna() & target.ge(1)
eligible = df.loc[valid_target_mask].copy()
eligible["target_los_days"] = target.loc[valid_target_mask].astype("int64")
eligible["target_los_log1p"] = np.log1p(eligible["target_los_days"])
eligible["target_is_multiday"] = eligible["target_los_days"].gt(1).astype("int8")

target_summary = eligible["target_los_days"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
).to_frame("LOS days")
display(target_summary)
print(f"One-day stays: {eligible['target_los_days'].eq(1).mean():.1%}")
print(f"Stays longer than 30 days retained for review: {eligible['target_los_days'].gt(30).sum():,}")

## 2. Engineer scheduling-time features

Free-text labels are normalized for case, accents, and repeated whitespace. Missing clinical information remains explicit. Counts summarize the presence of secondary diagnoses and procedures without inventing codes. High-cardinality categories are handled later using training-only frequencies.

In [ ]:
def normalize_category_value(value):
    if pd.isna(value) or not str(value).strip():
        return MISSING_LABEL
    text = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", text).strip().upper()


def normalize_category(series):
    return series.map(normalize_category_value).astype("string")


def normalize_code(series):
    categories = normalize_category(series)
    normalized = categories.str.replace(r"[^A-Z0-9]", "", regex=True)
    return normalized.mask(categories.eq(MISSING_LABEL) | normalized.eq(""), MISSING_LABEL)


def documented(series):
    return series.notna() & series.astype("string").str.strip().ne("")


features = pd.DataFrame(index=eligible.index)
features["age_years"] = pd.to_numeric(eligible["age_years"], errors="coerce")
features["sexe"] = normalize_category(eligible["sexe"])
features["interv_type"] = normalize_category(eligible["interv_type"])
features["anesth_type"] = normalize_category(eligible["anesth_type"])
features["anesth_loco_reg"] = normalize_category(eligible["anesth_loco_reg"])
features["praticien"] = normalize_category(eligible["praticien"])
features["nom_chir"] = normalize_category(eligible["nom_chir"])

intervention_date = pd.to_datetime(eligible["date_inter"], errors="coerce")
features["intervention_year"] = intervention_date.dt.year.astype("Int16")
month = intervention_date.dt.month
weekday = intervention_date.dt.weekday
features["intervention_month_sin"] = np.sin(2 * np.pi * (month - 1) / 12)
features["intervention_month_cos"] = np.cos(2 * np.pi * (month - 1) / 12)
features["intervention_weekday_sin"] = np.sin(2 * np.pi * weekday / 7)
features["intervention_weekday_cos"] = np.cos(2 * np.pi * weekday / 7)
features["intervention_is_weekend"] = intervention_date.dt.weekday.ge(5).astype("int8")

secondary_diagnosis_columns = [c for c in [f"cim_assoc_{i}" for i in range(1, 6)] if c in eligible]
procedure_columns = [c for c in [f"ccam_{i}" for i in range(1, 5)] if c in eligible]
features["num_secondary_diagnoses"] = sum(
    documented(eligible[c]).astype("int8") for c in secondary_diagnosis_columns
).astype("int8")
features["num_ccam_codes"] = sum(
    documented(eligible[c]).astype("int8") for c in procedure_columns
).astype("int8")
features["anesth_type_missing"] = features["anesth_type"].eq(MISSING_LABEL).astype("int8")
features["anesth_loco_reg_missing"] = features["anesth_loco_reg"].eq(MISSING_LABEL).astype("int8")

if INCLUDE_PLANNED_CLINICAL_CODES:
    features["cim_diag_pr"] = normalize_code(eligible["cim_diag_pr"])
    features["cim_diag_family"] = features["cim_diag_pr"].where(
        features["cim_diag_pr"].eq(MISSING_LABEL),
        features["cim_diag_pr"].str.slice(0, 3),
    )
    features["ccam_1"] = normalize_code(eligible["ccam_1"])
    features["ccam_1_family"] = features["ccam_1"].where(
        features["ccam_1"].eq(MISSING_LABEL),
        features["ccam_1"].str.slice(0, 4),
    )

categorical_features = [
    "sexe", "interv_type", "anesth_type", "anesth_loco_reg",
    "praticien", "nom_chir",
]
if INCLUDE_PLANNED_CLINICAL_CODES:
    categorical_features += ["cim_diag_pr", "cim_diag_family", "ccam_1", "ccam_1_family"]

numeric_features = [c for c in features.columns if c not in categorical_features]
print(f"Prepared {len(numeric_features)} numeric and {len(categorical_features)} categorical features.")

## 3. Create patient-disjoint temporal splits

Training uses interventions before 2021, validation uses new patients first represented in 2021, and testing uses new patients first represented from 2022 onward. Later records for patients already assigned to an earlier period are excluded. This conservative rule prevents the same patient from appearing in multiple splits while preserving future-data evaluation.

In [ ]:
def patient_disjoint_temporal_split(patient_ids, event_dates):
    split = pd.Series("excluded_cross_period", index=patient_ids.index, dtype="string")

    train_mask = event_dates.lt(VALIDATION_START)
    split.loc[train_mask] = "train"
    train_patients = set(patient_ids.loc[train_mask].dropna())

    validation_period = event_dates.ge(VALIDATION_START) & event_dates.lt(TEST_START)
    validation_mask = validation_period & ~patient_ids.isin(train_patients)
    split.loc[validation_mask] = "validation"
    validation_patients = set(patient_ids.loc[validation_mask].dropna())

    test_period = event_dates.ge(TEST_START)
    earlier_patients = train_patients | validation_patients
    test_mask = test_period & ~patient_ids.isin(earlier_patients)
    split.loc[test_mask] = "test"
    return split


split = patient_disjoint_temporal_split(eligible["id_patient"], intervention_date)
retained_mask = split.isin(["train", "validation", "test"])

model_data = pd.DataFrame({
    "split": split.loc[retained_mask],
    "split_patient_id": eligible.loc[retained_mask, "id_patient"],
    "split_event_date": intervention_date.loc[retained_mask],
})
model_data = pd.concat(
    [
        model_data,
        features.loc[retained_mask],
        eligible.loc[retained_mask, ["target_los_days", "target_los_log1p", "target_is_multiday"]],
    ],
    axis=1,
)

print(f"Excluded cross-period repeat-patient cases: {(~retained_mask).sum():,}")
display(model_data["split"].value_counts().rename("cases").to_frame())

## 4. Group rare categories using training data only

Categories occurring fewer than `MIN_CATEGORY_COUNT` times in training become `__RARE__`. Validation/test-only values therefore become rare rather than leaking future frequency information into preprocessing. Missing values remain a separate explicit category.

In [ ]:
rare_summary = []
train_mask = model_data["split"].eq("train")
for column in categorical_features:
    train_counts = model_data.loc[train_mask, column].value_counts(dropna=False)
    frequent_values = set(train_counts[train_counts.ge(MIN_CATEGORY_COUNT)].index)
    rare_mask = (
        ~model_data[column].isin(frequent_values)
        & model_data[column].ne(MISSING_LABEL)
    )
    rare_summary.append({
        "feature": column,
        "training_categories": len(train_counts),
        "retained_categories": len(frequent_values),
        "rows_grouped_as_rare": int(rare_mask.sum()),
    })
    model_data.loc[rare_mask, column] = RARE_LABEL

display(pd.DataFrame(rare_summary).set_index("feature"))

## 5. Validate and export

The `split_*` columns are metadata, not predictors. They remain in the local Parquet file so teammates can reproduce and audit the split.

In [ ]:
split_patient_sets = {
    name: set(group["split_patient_id"])
    for name, group in model_data.groupby("split")
}
assert split_patient_sets["train"].isdisjoint(split_patient_sets["validation"])
assert split_patient_sets["train"].isdisjoint(split_patient_sets["test"])
assert split_patient_sets["validation"].isdisjoint(split_patient_sets["test"])
assert model_data["target_los_days"].ge(1).all()
assert not model_data[categorical_features].isna().any().any()
assert not set([
    "date_sortie", "duree_sejour_corrigee", "length_of_stay_days",
    "room_duration_min", "id_patient", "no_cas",
]).intersection(features.columns)

split_summary = model_data.groupby("split").agg(
    cases=("target_los_days", "size"),
    patients=("split_patient_id", "nunique"),
    median_los=("target_los_days", "median"),
    p90_los=("target_los_days", lambda values: values.quantile(0.9)),
    multiday_share=("target_is_multiday", "mean"),
)
display(split_summary.round(3))

model_data = model_data.reset_index(drop=True)
model_data.to_parquet(OUTPUT_FILE, index=False, engine="pyarrow")
print(f"Saved {len(model_data):,} rows and {model_data.shape[1]} columns to {OUTPUT_FILE}")

## 6. Model-ingestion example

Use `numeric_features + categorical_features` as the feature list. Keep encoding, imputation, and scaling inside a fitted model pipeline. Never include columns beginning with `split_`, and choose only the target required by the modeling strategy.

In [ ]:
FEATURE_COLUMNS = numeric_features + categorical_features
train = model_data[model_data["split"].eq("train")]
validation = model_data[model_data["split"].eq("validation")]
test = model_data[model_data["split"].eq("test")]

X_train = train[FEATURE_COLUMNS]
y_train = train["target_los_days"]
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)